# SQL Aggregations — Complete Reference

| Pattern | Technique |
|---------|----------|
| GROUP BY + NULL | NULL grouping behavior, COALESCE |
| HAVING vs WHERE | Pre-aggregate vs post-aggregate filter |
| GROUPING SETS / ROLLUP / CUBE | Multi-level aggregation in one query |
| Conditional aggregation | CASE inside SUM/COUNT/AVG |
| Window vs GROUP BY | When to use each |

**Mental model**: GROUP BY collapses rows with the same key into one output row. Aggregation functions (SUM, COUNT, AVG, MIN, MAX) compute values across the group. Window functions keep all rows but add a per-row aggregate value.

```
GROUP BY execution order:
  FROM → WHERE (pre-filter) → GROUP BY → HAVING (post-filter) → SELECT → ORDER BY
  
  WHERE:  filters rows BEFORE grouping   (use for row-level conditions)
  HAVING: filters groups AFTER grouping  (use for aggregate conditions)
```

## Visual Model

```
RAW DATA             GROUP BY region          ROLLUP (adds subtotals)
───────────          ──────────────────       ──────────────────────────
region  amount       region   total           region  product  total
East    100          East     350             East    A        100
East    250          West     200             East    B        250
West    200                                  East    NULL     350  ← subtotal
                                             West    C        200
                                             West    NULL     200  ← subtotal
                                             NULL    NULL     550  ← grand total

CONDITIONAL AGGREGATION (pivot with CASE inside SUM)
─────────────────────────────────────────────────────
SELECT
  region,
  SUM(CASE WHEN status='completed' THEN amount ELSE 0 END) AS completed,
  SUM(CASE WHEN status='cancelled' THEN amount ELSE 0 END) AS cancelled
FROM orders GROUP BY region;

WINDOW vs GROUP BY
───────────────────
GROUP BY: collapses rows         Window: keeps all rows
  region | total                   region | amount | region_total
  East   | 350                     East   | 100    | 350
  West   | 200                     East   | 250    | 350
                                   West   | 200    | 200
```

## Setup — Libraries and Config

In [ ]:
import sqlite3

print(f"sqlite3 version: {sqlite3.sqlite_version}")

def make_db():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript("""
        CREATE TABLE orders (
            order_id    INTEGER PRIMARY KEY,
            customer_id INTEGER,
            region      TEXT,
            product     TEXT,
            category    TEXT,
            amount      REAL,
            status      TEXT,
            order_date  TEXT
        );
        INSERT INTO orders VALUES
            (1, 101,'East','Widget','Electronics', 250,'completed','2024-01-10'),
            (2, 101,'East','Gadget','Electronics', 180,'completed','2024-01-15'),
            (3, 102,'East','Shirt', 'Clothing',     75,'cancelled', '2024-01-12'),
            (4, 102,'East','Shirt', 'Clothing',     90,'completed','2024-02-05'),
            (5, 103,'West','Drill', 'Tools',        320,'completed','2024-01-08'),
            (6, 103,'West','Drill', 'Tools',        320,'pending',  '2024-02-20'),
            (7, 104,'West','Widget','Electronics',  250,'completed','2024-02-14'),
            (8, 104,'West','Cable', 'Electronics',   45,'completed','2024-03-01'),
            (9, 105,'East',NULL,    'Misc',          60,'completed','2024-03-10'),
            (10,105,'East','Hat',   'Clothing',      35,'completed','2024-03-15'),
            (11,106,'North','Drill','Tools',        400,'completed','2024-01-20'),
            (12,106,'North','Saw',  'Tools',        280,'cancelled','2024-02-10'),
            (13,107,'South','Food', 'Food',          15,'completed','2024-01-05'),
            (14,107,'South','Food', 'Food',          22,'completed','2024-02-28'),
            (15,108,'South','Tool', 'Tools',        150,'completed','2024-03-20');
    """)
    conn.commit()
    return conn

def run(conn, sql, title=""):
    cur = conn.execute(sql)
    rows = cur.fetchall()
    if not rows:
        print(f"{title}: (no rows)")
        return
    cols = [d[0] for d in cur.description]
    col_w = [max(len(c), max(len(str(r[c])) for r in rows)) for c in cols]
    sep = "  ".join("-" * w for w in col_w)
    header = "  ".join(c.ljust(w) for c, w in zip(cols, col_w))
    if title:
        print(f"\n=== {title} ===")
    print(header)
    print(sep)
    for r in rows:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(cols, col_w)))

conn = make_db()
print("Database ready.")

## Decision Map — Aggregation Patterns

```
What do you need?
│
├─ Simple totals per group?               → GROUP BY + SUM/COUNT/AVG
├─ Filter groups by aggregate value?      → HAVING (not WHERE)
├─ Filter rows before grouping?           → WHERE
├─ Subtotals + grand total in one query?  → GROUP BY ... WITH ROLLUP
├─ All combinations of group levels?      → CUBE
├─ Specific set of grouping combinations? → GROUPING SETS
├─ Pivot (status as columns)?             → CASE inside SUM/COUNT
├─ Ratio within group vs collapsing?      → Window function (PARTITION BY)
└─ Percentile / median?                   → NTILE or ORDER-SET aggregates

NULL in GROUP BY:
  NULL is treated as its own group key — rows with NULL product grouped together
  SELECT product, COUNT(*) FROM orders GROUP BY product
  → NULL appears as a separate group row

NULL in aggregates:
  COUNT(*) counts all rows including NULLs
  COUNT(col) counts only non-NULL values of col
  SUM/AVG/MIN/MAX ignore NULLs automatically
```

## Pattern 1 — GROUP BY + NULL Handling

In [ ]:
# NULL is a valid GROUP BY key — rows with NULL group together
# COUNT(*) vs COUNT(col) behavior with NULLs is a common interview trap

run(conn, """
    SELECT
        product,
        COUNT(*)       AS row_count,
        COUNT(product) AS non_null_count,
        SUM(amount)    AS total_amount
    FROM orders
    GROUP BY product
    ORDER BY product
""", "NULL grouping: NULL becomes its own group")

print("\nNote: product=NULL row shows COUNT(*) > COUNT(product)")
print("COUNT(*) counts rows; COUNT(col) counts non-NULL values of that column")

# Handle NULLs with COALESCE in GROUP BY
run(conn, """
    SELECT
        COALESCE(product, 'UNKNOWN') AS product,
        COUNT(*)    AS orders,
        SUM(amount) AS revenue
    FROM orders
    GROUP BY COALESCE(product, 'UNKNOWN')
    ORDER BY revenue DESC
""", "COALESCE NULL to UNKNOWN label")

# Multi-column GROUP BY
run(conn, """
    SELECT
        region,
        category,
        COUNT(*)    AS orders,
        SUM(amount) AS revenue,
        ROUND(AVG(amount), 2) AS avg_order
    FROM orders
    WHERE status = 'completed'
    GROUP BY region, category
    ORDER BY region, revenue DESC
""", "Multi-column GROUP BY: region x category")

## Pattern 2 — HAVING vs WHERE

In [ ]:
# WHERE filters BEFORE aggregation (row-level conditions)
# HAVING filters AFTER aggregation (group-level conditions)
# Performance: WHERE is always faster — reduces rows fed to GROUP BY

# Wrong: using HAVING for a row-level filter (works but slower)
print("=== BAD: HAVING for row-level filter ===")
run(conn, """
    SELECT region, SUM(amount) AS total
    FROM orders
    GROUP BY region
    HAVING status = 'completed'   -- ERROR: status not in GROUP BY or aggregate
""", "HAVING on non-aggregate")

# Actually SQLite allows this but it's non-standard; correct approach:
print("\n=== CORRECT: WHERE for row conditions, HAVING for group conditions ===")
run(conn, """
    SELECT region, SUM(amount) AS total
    FROM orders
    WHERE status = 'completed'    -- filter rows BEFORE grouping
    GROUP BY region
    HAVING SUM(amount) > 300      -- filter groups AFTER grouping
    ORDER BY total DESC
""", "WHERE (row filter) + HAVING (group filter)")

# HAVING with multiple aggregate conditions
run(conn, """
    SELECT
        customer_id,
        COUNT(*)          AS order_count,
        SUM(amount)       AS total_spend,
        MIN(order_date)   AS first_order
    FROM orders
    WHERE status != 'cancelled'
    GROUP BY customer_id
    HAVING COUNT(*) >= 2             -- at least 2 orders
       AND SUM(amount) > 200         -- and total > 200
    ORDER BY total_spend DESC
""", "Multi-condition HAVING: active customers with >= 2 orders")

## Pattern 3 — GROUPING SETS / ROLLUP / CUBE

In [ ]:
# ROLLUP: hierarchical subtotals (e.g., region > category > grand total)
# CUBE: all possible combinations of group keys
# GROUPING SETS: explicit list of grouping combinations
# SQLite doesn't support ROLLUP/CUBE natively — simulate with UNION ALL

# SQLite simulation of ROLLUP(region, category)
run(conn, """
    -- Level 1: region + category detail
    SELECT region, category, SUM(amount) AS total, 'detail' AS level
    FROM orders WHERE status='completed' GROUP BY region, category
    UNION ALL
    -- Level 2: region subtotals
    SELECT region, NULL, SUM(amount), 'region_subtotal'
    FROM orders WHERE status='completed' GROUP BY region
    UNION ALL
    -- Level 3: grand total
    SELECT NULL, NULL, SUM(amount), 'grand_total'
    FROM orders WHERE status='completed'
    ORDER BY region, category
""", "ROLLUP simulation: region > category > grand total")

# Standard SQL ROLLUP (works in PostgreSQL, BigQuery, Snowflake, MySQL 8+)
print("""
Standard SQL (PostgreSQL/BigQuery syntax):

  SELECT region, category, SUM(amount)
  FROM orders
  GROUP BY ROLLUP(region, category);
  -- Produces: (region,category), (region,NULL), (NULL,NULL)

  SELECT region, category, SUM(amount)
  FROM orders
  GROUP BY CUBE(region, category);
  -- Produces: (region,category), (region,NULL), (NULL,category), (NULL,NULL)

  SELECT region, category, SUM(amount)
  FROM orders
  GROUP BY GROUPING SETS ((region, category), (region), ());
  -- Exactly the ROLLUP set, explicitly named

  -- Distinguish NULL-from-ROLLUP vs real NULL:
  SELECT region, GROUPING(region) AS is_rollup_null, SUM(amount)
  FROM orders GROUP BY ROLLUP(region);
  -- GROUPING() = 1 means the NULL came from ROLLUP, not the data
""")

## Pattern 4 — Conditional Aggregation

In [ ]:
# CASE inside aggregate functions = conditional aggregation
# Enables pivot (turn row values into columns) without joins

# Pivot: status as columns
run(conn, """
    SELECT
        region,
        COUNT(*) AS total_orders,
        SUM(CASE WHEN status='completed' THEN 1 ELSE 0 END) AS completed,
        SUM(CASE WHEN status='cancelled' THEN 1 ELSE 0 END) AS cancelled,
        SUM(CASE WHEN status='pending'   THEN 1 ELSE 0 END) AS pending,
        ROUND(100.0 * SUM(CASE WHEN status='completed' THEN 1 ELSE 0 END)
              / COUNT(*), 1) AS completion_pct
    FROM orders
    GROUP BY region
    ORDER BY region
""", "Conditional count pivot: status by region")

# Revenue pivot: category as columns
run(conn, """
    SELECT
        region,
        ROUND(SUM(CASE WHEN category='Electronics' THEN amount ELSE 0 END), 2) AS electronics,
        ROUND(SUM(CASE WHEN category='Clothing'    THEN amount ELSE 0 END), 2) AS clothing,
        ROUND(SUM(CASE WHEN category='Tools'       THEN amount ELSE 0 END), 2) AS tools,
        ROUND(SUM(CASE WHEN category='Food'        THEN amount ELSE 0 END), 2) AS food,
        ROUND(SUM(amount), 2) AS total
    FROM orders
    WHERE status = 'completed'
    GROUP BY region
    ORDER BY total DESC
""", "Revenue pivot: category as columns")

# Conditional AVG: average only for a subset
run(conn, """
    SELECT
        category,
        COUNT(*) AS orders,
        ROUND(AVG(amount), 2) AS avg_all,
        ROUND(AVG(CASE WHEN status='completed' THEN amount END), 2) AS avg_completed,
        ROUND(AVG(CASE WHEN amount > 200 THEN amount END), 2) AS avg_large_orders
    FROM orders
    GROUP BY category
    ORDER BY category
""", "Conditional AVG (NULLs excluded by aggregate functions)")

## Pattern 5 — Window vs GROUP BY

In [ ]:
# GROUP BY: collapses rows into one per group
# Window: keeps all rows, adds aggregate as a new column
# Use window when you need both row detail AND group aggregate

# GROUP BY: one row per region
run(conn, """
    SELECT region, SUM(amount) AS region_total
    FROM orders
    WHERE status = 'completed'
    GROUP BY region
    ORDER BY region
""", "GROUP BY: one row per region (row detail lost)")

# Window: keeps all rows, adds region total per row
run(conn, """
    SELECT
        order_id,
        region,
        product,
        amount,
        SUM(amount) OVER (PARTITION BY region) AS region_total,
        ROUND(100.0 * amount / SUM(amount) OVER (PARTITION BY region), 1) AS pct_of_region
    FROM orders
    WHERE status = 'completed'
    ORDER BY region, amount DESC
""", "Window: all rows + region total (row detail preserved)")

# Combined: GROUP BY summary JOINED to row-level detail
run(conn, """
    WITH region_stats AS (
        SELECT region,
               COUNT(*) AS region_orders,
               ROUND(SUM(amount), 2) AS region_total
        FROM orders WHERE status='completed'
        GROUP BY region
    )
    SELECT
        o.order_id, o.region, o.product, o.amount,
        rs.region_orders, rs.region_total
    FROM orders o
    JOIN region_stats rs ON o.region = rs.region
    WHERE o.status = 'completed'
    ORDER BY o.region, o.amount DESC
    LIMIT 8
""", "CTE pattern: GROUP BY summary joined to row detail")

## Full Decision Map

```
SQL AGGREGATION GUIDE
──────────────────────
Filter rows first?         → WHERE (before GROUP BY)
Filter groups?             → HAVING (after GROUP BY)
Collapse rows to groups?   → GROUP BY
Keep rows, add aggregate?  → Window function OVER(PARTITION BY)
Subtotals + grand total?   → ROLLUP (or UNION ALL simulation)
All group combinations?    → CUBE
Specific combinations?     → GROUPING SETS
Turn rows into columns?    → CASE inside SUM/COUNT

NULL RULES
  GROUP BY NULL: NULLs form their own group (one group per NULL cluster)
  COUNT(*): includes NULLs  |  COUNT(col): excludes NULLs
  SUM/AVG/MIN/MAX: NULLs ignored automatically
  ROLLUP NULL vs data NULL: use GROUPING() function to distinguish

PERFORMANCE
  WHERE > HAVING (fewer rows to aggregate)
  Composite index on (group_col, filter_col) helps large GROUP BY queries
  Push GROUP BY into CTE before joining other tables
  DISTINCT inside COUNT is expensive — deduplicate in CTE first

INTERVIEW TRAPS
  SELECT cols not in GROUP BY or aggregate → error (most DBs)
  WHERE aggregate_col > X → error → use HAVING
  COUNT(*) vs COUNT(col) difference → critical
  AVG ignores NULLs — can differ from SUM/COUNT manually
```

## Cheat Sheet

```sql
-- Basic aggregation
SELECT col, COUNT(*), SUM(amt), AVG(amt), MIN(amt), MAX(amt)
FROM t WHERE status='active'
GROUP BY col
HAVING COUNT(*) > 5
ORDER BY SUM(amt) DESC;

-- NULL in aggregate
COUNT(*)         -- all rows incl. NULLs
COUNT(col)       -- non-NULL only
COUNT(DISTINCT col)  -- unique non-NULL values
COALESCE(SUM(amt), 0)  -- NULL → 0 when no rows match

-- Conditional aggregation (pivot)
SELECT region,
  SUM(CASE WHEN status='completed' THEN amount ELSE 0 END) AS completed_rev,
  SUM(CASE WHEN status='cancelled' THEN 1       ELSE 0 END) AS cancelled_cnt
FROM orders GROUP BY region;

-- ROLLUP (PostgreSQL/BigQuery/MySQL 8+)
SELECT region, category, SUM(amount)
FROM orders GROUP BY ROLLUP(region, category);

-- CUBE (all combinations)
SELECT region, category, SUM(amount)
FROM orders GROUP BY CUBE(region, category);

-- GROUPING SETS (specific combinations)
SELECT region, category, SUM(amount)
FROM orders
GROUP BY GROUPING SETS ((region, category), (region), ());

-- Window: keep rows, add group aggregate
SELECT *, SUM(amount) OVER (PARTITION BY region) AS region_total
FROM orders;

-- ROLLUP simulation (SQLite)
SELECT region, category, SUM(amount) FROM orders GROUP BY region, category
UNION ALL SELECT region, NULL, SUM(amount) FROM orders GROUP BY region
UNION ALL SELECT NULL,   NULL, SUM(amount) FROM orders;
```

## Summary Map

```
SQL AGGREGATIONS — ONE-PAGE SUMMARY
─────────────────────────────────────

EXECUTION ORDER
  FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY
  → WHERE reduces rows before grouping (always prefer WHERE over HAVING for rows)

KEY FUNCTIONS
  COUNT(*) / COUNT(col) / COUNT(DISTINCT col)
  SUM / AVG / MIN / MAX  — all ignore NULLs
  GROUP_CONCAT(col)      — string aggregation (SQLite)

MULTI-LEVEL AGGREGATION
  ROLLUP(a, b)   → (a,b), (a,NULL), (NULL,NULL)   — hierarchical
  CUBE(a, b)     → (a,b), (a,NULL), (NULL,b), (NULL,NULL) — all combos
  GROUPING SETS  → user-specified subset of CUBE combinations
  SQLite workaround: UNION ALL of individual GROUP BYs

CONDITIONAL AGGREGATION
  CASE inside SUM/COUNT/AVG → pivot rows to columns
  No separate PIVOT syntax needed in most engines

WINDOW vs GROUP BY
  GROUP BY: N rows → M groups (rows collapsed)
  Window:   N rows → N rows + new aggregate column
  Choose window when you need row detail AND group context

INTERVIEW SIGNALS
  ✓ WHERE is row filter (pre-agg), HAVING is group filter (post-agg)
  ✓ COUNT(*) vs COUNT(col) NULL difference
  ✓ Conditional aggregation replaces PIVOT syntax
  ✓ Window function preserves row count; GROUP BY collapses
  ✓ ROLLUP for hierarchical subtotals with one query
```